# WS 8 - MACE Fine Tuning

**Machine Learning in Computational Physics** · University of Vienna

---

## Learning Objectives

In this tutorial, you will learn how to improve MLIP models by using iterative training and active learning. We illustrate these training workflows on MACE, but they are broadly applicable to all MLIPs. We will also showcase the state-of-the-art [foundational models](https://matbench-discovery.materialsproject.org/) - the latest development in the field of MLIPs. These models are trained on massive training sets of [inorganic](https://doi.org/10.48550/arXiv.2401.00096) and [organic](https://doi.org/10.48550/arXiv.2312.15211) databases and show a great deal of `out-of-the-box` MD stability in an extensive variety of [applications](https://doi.org/10.48550/arXiv.2401.00096). We will discuss [fine-tunning](https://doi.org/10.48550/arXiv.2405.20217) which is an actively-researched technique to tweak these foundational models to new systems (out of training) and/or new levels of reference theory.


1. **Iterative Training: improving stability and accuracy**
2. **Active learning: committee models**
3. **Foundational models**


In [ ]:
# Standard imports
import numpy as np
import matplotlib.pyplot as plt
import os
from pathlib import Path
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Locate repo root (contains pyproject.toml) — works in Docker, VS Code, and local Jupyter
_p = Path().resolve()
while not (_p / 'pyproject.toml').exists() and _p != _p.parent:
    _p = _p.parent
DATA_DIR = _p / 'data'

os.makedirs(_p / "config", exist_ok=True)
os.makedirs(_p / "MACE_models", exist_ok=True)
os.makedirs(DATA_DIR / "mace_moldyn", exist_ok=True)

## 1. Iterative Training

### 1.1 MD with a smaller MACE model

The model we trained in our previous tutorial was already stable in MD and quite accurate with little training. This is both because MACE models are smooth and very accurate but also because the task of simulating a single molecule for a few picoseconds is not all that difficult. In general, in real research applications, achieving MD stability and accuracy is not always straightforward from the get-go. Models can be improved through iterative training and active learning which expands the training data to fix errors on the model's potential energy surface.

To illustrate these concepts in practice, let's first train a less accurate MACE by reducing a lot the model size and amount of training data: we will make a model with just 20 training configurations. Note that a larger model would probably work out of the box for this example.

In [ ]:
from ase.io import read, write
from tblite.ase import TBLite
import yaml

# choose your device here.
#device = 'cpu'
device =  'cuda' 

#test data from WS7
test_file=str(DATA_DIR / 'mace_data' / 'solvent_xtb_test.xyz')

#xTB training database from WS7
db = read(DATA_DIR / 'mace_data' / 'solvent_xtb.xyz', ':')

train_file=str(DATA_DIR / 'mace_data' / 'solvent_xtb_train_20.xyz')
write(train_file, db[:23]) #first 20 configs plus the 3 E0s will be used for training (parameter optimization)
valid_file=str(DATA_DIR / 'mace_data' / 'solvent_xtb_valid_20.xyz')
write(valid_file, db[23:43]) #next 20 configs will be used for validation (independent, stop condition)




In [ ]:
%%writefile config/config-03.yml

model: "MACE"
num_interactions: 2 
num_channels: 32
max_L: 0
r_max: 4.0
max_ell: 2
name: "mace02_com1"
model_dir: "MACE_models"
log_dir: "MACE_models"
checkpoints_dir: "MACE_models"
results_dir: "MACE_models"
energy_key: "energy_xtb"
forces_key: "forces_xtb"
E0s: "isolated"
batch_size: 10
max_num_epochs: 200
stage_two: True
seed: 123

In [ ]:
dev = f'device: {device}'
train = f'train_file: {train_file}'
test = f'test_file: {test_file}'
valid = f'valid_file: {valid_file}'

%store dev >>"config/config-03.yml"
%store train >>"config/config-03.yml"
%store test >>"config/config-03.yml"
%store valid >>"config/config-03.yml"

In [ ]:
import warnings
warnings.filterwarnings("ignore")
from mace.cli.run_train import main as mace_run_train_main
import sys
import logging

def train_mace(config_file_path):
    logging.getLogger().handlers.clear()
    sys.argv = ["program", "--config", config_file_path]
    mace_run_train_main()

In [ ]:
train_mace("config/config-03.yml")

In [ ]:
#remove checkpoints since they may cause errors on retraining a model with the same name but a different architecture
import glob
import os
for file in glob.glob("MACE_models/*_run-*.model"):
    os.remove(file)
for file in glob.glob("MACE_models/*.pt"):
    os.remove(file)

Notice, we are getting substantially larger errors than in the last workshop. Now, let's run some dynamics:

In [ ]:
from ase.io import read, write
from ase import units
from ase.md.langevin import Langevin
from ase.md.velocitydistribution import Stationary, ZeroRotation, MaxwellBoltzmannDistribution

import random
import os
import time
import numpy as np
import pylab as pl
from IPython import display

def simpleMD(init_conf, temp, calc, fname, s, T):
    init_conf.set_calculator(calc)

    #initialize the temperature
    random.seed(701) #just making sure the MD failure is reproducible
    MaxwellBoltzmannDistribution(init_conf, temperature_K=300) #initialize temperature at 300
    Stationary(init_conf)
    ZeroRotation(init_conf)

    dyn = Langevin(init_conf, 1.0*units.fs, temperature_K=temp, friction=0.1) #drive system to desired temperature

    %matplotlib inline

    time_fs = []
    temperature = []
    energies = []

    #remove previously stored trajectory with the same name
    if os.path.exists(fname):
        os.remove(fname)

    fig, ax = pl.subplots(2, 1, figsize=(6,6), sharex='all', gridspec_kw={'hspace': 0, 'wspace': 0})

    def write_frame():
            dyn.atoms.write(fname, append=True)
            time_fs.append(dyn.get_time()/units.fs)
            temperature.append(dyn.atoms.get_temperature())
            energies.append(dyn.atoms.get_potential_energy()/len(dyn.atoms))

            ax[0].plot(np.array(time_fs), np.array(energies), color="b")
            ax[0].set_ylabel('E (eV/atom)')

            # plot the temperature of the system as subplots
            ax[1].plot(np.array(time_fs), temperature, color="r")
            ax[1].set_ylabel('T (K)')
            ax[1].set_xlabel('Time (fs)')

            display.clear_output(wait=True)
            display.display(pl.gcf())
            time.sleep(0.01)

    dyn.attach(write_frame, interval=s)
    t0 = time.time()
    dyn.run(T)
    t1 = time.time()
    print("MD finished in {0:.2f} minutes!".format((t1-t0)/60))

In [ ]:
from aseMolec import extAtoms as ea
#let us start with a single molecule
init_conf = ea.sel_by_info_val(read(DATA_DIR / 'mace_data/solvent_molecs.xyz', ':'), 'Nmols', 1)[0].copy()

#we can use MACE as a calculator in ASE!
from mace.calculators import MACECalculator
mace_calc = MACECalculator(model_paths=[str(_p / 'MACE_models/mace02_com1_stagetwo_compiled.model')], device=device)

fname = str(DATA_DIR / 'mace_moldyn' / 'mace02_md.xyz')
simpleMD(init_conf, temp=1200, calc=mace_calc, fname=fname, s=10, T=2000)

Depending on the random seed for the MD, you may see different things. At first sight, the energy against time doesn't look too bad. Let's look at the trajectory.

In [ ]:
from ase.io import read, write

traj = read(DATA_DIR / 'mace_moldyn' / 'mace02_md.xyz', ':')

from weas_widget import WeasWidget
viewer = WeasWidget()
viewer.from_ase(traj)
viewer.avr.model_style = 1
viewer

If you go to the end of the trajectory, you should find that the bond angles are actually very strange - it looks unphysical.

**Task for you**

<div class="alert alert-block alert-info">

Sometimes, when the model is very bad you may find the simulation exploding. Try for yourself, reduce the size of the model (e.g. number of channels) and see if a small model will generate exploding simulations. Why do simulations explode with bad models? What happens physically?
</div>

### 1.2 Identify the problem and expand training

Something doesn't look right with the hydrogen atoms. Let's re-evaluate the first 100 steps from the trajectory on the reference xTB potential energy surface and then plot it against MACE energy.

In [ ]:
from tblite.ase import TBLite
from ase.io import read, write
from tqdm import tqdm
from mace.calculators import MACECalculator
mace_calc = MACECalculator(model_paths=['MACE_models/mace02_com1_stagetwo_compiled.model'], device=device)

xtb_calc = TBLite(method="GFN2-xTB",verbosity=0)

#compute true reference XTB values
traj = read(DATA_DIR / 'mace_moldyn/mace02_md.xyz', ':')
for at in tqdm(traj[:100]):
    at.calc = xtb_calc
    at.info['energy_xtb'] = at.get_potential_energy()
    at.arrays['forces_xtb'] = at.get_forces()
    at.calc = mace_calc
    at.info['energy_mace'] = at.get_potential_energy()


write(DATA_DIR / 'mace_data/mace02_md_100_xtb.xyz', traj[:100]) #save full result

In [ ]:
import numpy as np
from aseMolec import extAtoms as ea
from matplotlib import pyplot as plt

traj = read(DATA_DIR / 'mace_data/mace02_md_100_xtb.xyz', ':')
plt.plot(np.arange(len(traj)), ea.get_prop(traj, 'info', 'energy_xtb', peratom=True), label='XTB')
plt.plot(np.arange(len(traj)), ea.get_prop(traj, 'info', 'energy_mace', peratom=True), label='MACE')
plt.legend()
plt.xlabel('Time (fs)')
plt.ylabel('Total Energy per Atom (eV)')

Indeed, in the second half of the trajectory, the xTB energy diverges because MACE finds unphysical configurations. With older potentials, at this point the MD would explode, but because MACE is much smoother the simulation keeps going, albeit generating the wrong dynamics. 

Let's take three of these unphysical configs, add them back to the training set and refit a new model. This is called iterative training:
![alt text](https://github.com/imagdau/Tutorials/blob/main/figures/iterative_training.png?raw=1)

In [ ]:
db = read(DATA_DIR / 'mace_data/solvent_xtb_train_20.xyz', ':')
db += traj[40:100:20] #add failed configs to the training set
write(DATA_DIR / 'mace_data/solvent_xtb_train_23_gen1.xyz', db)

**Task for you**

<div class="alert alert-block alert-info">

Above, we take three equidistant trajectories between frames 40 and 100. A good exercise is to try to pick the configs that have the largest error on forces and energies. Try coding it yourself.
</div>



### 1.3 Train a new MACE model and run MD again

In [ ]:
%%writefile config/config-04.yml

model: "MACE"
num_interactions: 2 
num_channels: 32
max_L: 0
r_max: 4.0
max_ell: 2
name: "mace02_com1_gen1"
model_dir: "MACE_models"
log_dir: "MACE_models"
checkpoints_dir: "MACE_models"
results_dir: "MACE_models"
energy_key: "energy_xtb"
forces_key: "forces_xtb"
E0s: "isolated"
batch_size: 10
max_num_epochs: 200
stage_two: True
seed: 123



In [ ]:
dev = f'device: {device}'
%store dev >>"config/config-04.yml"

train_file=str(DATA_DIR / 'mace_data' / 'solvent_xtb_train_23_gen1.xyz')
valid_file=str(DATA_DIR / 'mace_data' / 'solvent_xtb_valid_20.xyz')
test_file=str(DATA_DIR / 'mace_data' / 'solvent_xtb_test.xyz')
train = f'train_file: {train_file}'
test = f'test_file: {test_file}'
valid = f'valid_file: {valid_file}'
%store train >>"config/config-04.yml"
%store test >>"config/config-04.yml"
%store valid >>"config/config-04.yml"


In [ ]:
train_mace("config/config-04.yml")

In [ ]:
#remove checkpoints since they may cause errors on retraining a model with the same name but a different architecture
import glob
import os
for file in glob.glob("MACE_models/*_run-*.model"):
    os.remove(file)
for file in glob.glob("MACE_models/*.pt"):
    os.remove(file)

For some reason the energy error on the training set is now huge - can you work out why this is?

What does this imply about how we do iterative training?

In [ ]:
from mace.calculators import MACECalculator

mace_calc = MACECalculator(model_paths=['MACE_models/mace02_com1_gen1_stagetwo_compiled.model'], device='cuda')
init_conf = ea.sel_by_info_val(read(DATA_DIR / 'mace_data/solvent_molecs.xyz',':'), 'Nmols', 1)[0].copy()
simpleMD(init_conf, temp=1200, calc=mace_calc, fname=DATA_DIR / 'mace_moldyn/mace02_md_gen1.xyz', s=10, T=2000)

In [ ]:
from weas_widget import WeasWidget

traj = read(DATA_DIR / 'mace_moldyn/mace02_md_gen1.xyz', ':')
w = WeasWidget()
w.from_ase(traj)
w.avr.model_style = 1
w

Great! The dynamics is already looking better, however it can be difficult to tell if it is really correct or not.

**Task for you**
<div class="alert alert-block alert-info">
    
- Compute the true xTB energies and check if they agree with the MACE energies predicted along the trajectory
- Compute radial distribution functions (RDFs) and compare to a ground truth RDF (mace_data/xtb_md.xyz). **Tip**: Look at previous workshops on how to calculate RDFs.
- Compare the quality of the iterative training with 3 equidistant geometries and the 3 with the highest error (see task before)
</div>

This is an arduous process, because we need to carefully investigate the trajectories and decide which configs to add back to training. If the ground truth is expensive to compute (DFT, QM) this becomes a very slow process. We could instead approximate the errors (e.g. via a committee of models) on the fly and select configs which are not well predicted. This is called active learning and has the benefit of somewhat automating the process and avoiding the calculation of ground truths to assess whether new data is required.

## 2. Active Learning with MACE

### 2.1 Preparing a committee of models

We can **compute** errors by evaluating the reference energy and forces (in our case xTB) and computing the difference to MACE predictions. In real research applications, this can be very expensive to evaluate depending on the reference level of theory. Alternatively, we can **estimate** errors based on a committee of models. Let's train a committee of MACE models by adding some randomness to the optimization process. We can achieve this by changing the `--seed`. We already have a model, we will fit two more, on the same data:

In [ ]:
%%writefile config/config-05.yml

model: "MACE"
num_interactions: 2 
num_channels: 32
max_L: 0
r_max: 4.0
max_ell: 2
name: "mace02_com2"
model_dir: "MACE_models"
log_dir: "MACE_models"
checkpoints_dir: "MACE_models"
results_dir: "MACE_models"
energy_key: "energy_xtb"
forces_key: "forces_xtb"
E0s: "isolated"
batch_size: 10
max_num_epochs: 200
stage_two: True
seed: 345


In [ ]:
dev = f'device: {device}'
%store dev >>"config/config-05.yml"

train_file=str(DATA_DIR / 'mace_data' / 'solvent_xtb_train_20.xyz')
valid_file=str(DATA_DIR / 'mace_data' / 'solvent_xtb_valid_20.xyz')
test_file=str(DATA_DIR / 'mace_data' / 'solvent_xtb_test.xyz')

train = f'train_file: {train_file}'
test = f'test_file: {test_file}'
valid = f'valid_file: {valid_file}'
%store train >>"config/config-05.yml"
%store test >>"config/config-05.yml"
%store valid >>"config/config-05.yml"

In [ ]:
%%writefile config/config-06.yml

model: "MACE"
num_interactions: 2
num_channels: 32
max_L: 0
r_max: 4.0
max_ell: 2
name: "mace02_com3"
model_dir: "MACE_models"
log_dir: "MACE_models"
checkpoints_dir: "MACE_models"
results_dir: "MACE_models"
energy_key: "energy_xtb"
forces_key: "forces_xtb"
E0s: "isolated"
batch_size: 10
max_num_epochs: 200
stage_two: True
seed: 567


In [ ]:
dev = f'device: {device}'
%store dev >>"config/config-06.yml"
train_file=str(DATA_DIR / 'mace_data' / 'solvent_xtb_train_20.xyz')
valid_file=str(DATA_DIR / 'mace_data' / 'solvent_xtb_valid_20.xyz')
test_file=str(DATA_DIR / 'mace_data' / 'solvent_xtb_test.xyz')
train = f'train_file: {train_file}'
test = f'test_file: {test_file}'
valid = f'valid_file: {valid_file}'
%store train >>"config/config-06.yml"
%store test >>"config/config-06.yml"
%store valid >>"config/config-06.yml"

In [ ]:
train_mace("config/config-05.yml")
train_mace("config/config-06.yml")

In [ ]:
#remove checkpoints since they may cause errors on retraining a model with the same name but a different architecture
import glob
import os
for file in glob.glob("MACE_models/*_run-*.model"):
    os.remove(file)
for file in glob.glob("MACE_models/*.pt"):
    os.remove(file)

Perfect, we have two new models. Let's start by testing the commitee on the first 100 frames of the first trajectory we generated. The `MACECalculator` can conveniently take a list of calculators as input and will compute separate energies from each calculator.

In [ ]:
import numpy as np
from ase.io import read
from aseMolec import extAtoms as ea
from matplotlib import pyplot as plt
from mace.calculators import MACECalculator
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")

model_paths = ['MACE_models/mace02_com1_stagetwo_compiled.model',
               'MACE_models/mace02_com2_stagetwo_compiled.model',
               'MACE_models/mace02_com3_stagetwo_compiled.model',]
mace_calcs = MACECalculator(model_paths=model_paths, device=device)

traj = read(DATA_DIR / 'mace_data' /'mace02_md_100_xtb.xyz', ':')
for at in tqdm(traj):
    at.calc = mace_calcs
    at.info['average_mace_energy'] = at.get_potential_energy()
    at.info['energy_mace_1'] = at.calc.results["energy_comm"][0]  
    at.info['energy_mace_2'] = at.calc.results["energy_comm"][1]
    at.info['energy_mace_3'] = at.calc.results["energy_comm"][2]
    at.info['variance'] = at.calc.results["energy_var"]
    at.info['true_error'] = np.abs(at.calc.results["energy"] - at.info['energy_xtb'])     

#Let's check the energies of the MACE committee vs XTB energy
plt.figure(figsize=(12,6))
plt.subplot(1,2,1)

plt.plot(np.arange(len(traj)), ea.get_prop(traj, 'info', 'energy_xtb', peratom=True), label='XTB');
plt.plot(np.arange(len(traj)), ea.get_prop(traj, 'info', 'energy_mace_1', peratom=True), label='MACE_1');
plt.plot(np.arange(len(traj)), ea.get_prop(traj, 'info', 'energy_mace_2', peratom=True), label='MACE_2');
plt.plot(np.arange(len(traj)), ea.get_prop(traj, 'info', 'energy_mace_3', peratom=True), label='MACE_3');
plt.legend()
plt.xlabel('Time (fs)');
plt.ylabel('Energy per Atom (eV)');


fig, ax1 = plt.subplots()
ax1.plot(np.arange(len(traj)), ea.get_prop(traj, 'info', 'variance', peratom=False), label='committee variance', color='tab:blue')

ax1.set_xlabel('time (fs)')
ax1.set_ylabel('committee energy variance', color='tab:blue')

plt.show()

Notice how the variance (disagreement between models) increases around the same config where the true error with respect to xTB diverges. This is good news because it indicates the variance is a good proxy for true error.

**Task for you**
<div class="alert alert-block alert-info">

- Make a correlation plot between true error and estimated error. How well does the estimation perform?
- Try a commitee of models with both different seeds and trained on different selections of data. Does it perform better? Is the correlation better?
</div>

Now we can run dynamics with a commitee of models and monitor the variance in the energy prediction. Because xTB is cheap enough we can also compare that variance with the true error. Do they correlate?

### 2.2 Running MD with the MACE committee

In [ ]:
from aseMolec import extAtoms as ea
from ase import units
from ase.md.langevin import Langevin
from ase.md.velocitydistribution import Stationary, ZeroRotation, MaxwellBoltzmannDistribution
from ase.io import read, write

import random
import numpy as np
import time
import pylab as pl
from IPython import display

from tblite.ase import TBLite
from mace.calculators import MACECalculator

model_paths = ['MACE_models/mace02_com1_stagetwo_compiled.model',
               'MACE_models/mace02_com2_stagetwo_compiled.model',
               'MACE_models/mace02_com3_stagetwo_compiled.model']

xtb_calc = TBLite(method="GFN2-xTB",verbosity=0)
mace_calc = MACECalculator(model_paths=model_paths, device=device)

init_conf = ea.sel_by_info_val(read(DATA_DIR / 'mace_data' / 'solvent_molecs.xyz', ':'), 'Nmols', 1)[0].copy()
init_conf.calc = mace_calc

#initialize the temperature
np.random.seed(701)
MaxwellBoltzmannDistribution(init_conf, temperature_K=300)
Stationary(init_conf)
ZeroRotation(init_conf)

dyn = Langevin(init_conf, 1*units.fs, temperature_K=1200, friction=0.1)

%matplotlib inline

time_fs = []
temperature = []
energies_1 = []
energies_2 = []
energies_3 = []
variances = []
xtb_energies = []
true_errors = []

!rm -rfv moldyn/mace02_md_committee.xyz
fig, ax = pl.subplots(4, 1, figsize=(8,8), sharex='all', gridspec_kw={'hspace': 0, 'wspace': 0})


def write_frame():
        at = dyn.atoms.copy()
        at.calc = xtb_calc
        xtb_energy = at.get_potential_energy()

        dyn.atoms.write(DATA_DIR / 'mace_moldyn' / 'mace02_md_committee.xyz', append=True, write_results=False)
        time_fs.append(dyn.get_time()/units.fs)
        temperature.append(dyn.atoms.get_temperature())
        energies_1.append(dyn.atoms.calc.results["energy_comm"][0]/len(dyn.atoms))
        energies_2.append(dyn.atoms.calc.results["energy_comm"][1]/len(dyn.atoms))
        energies_3.append(dyn.atoms.calc.results["energy_comm"][2]/len(dyn.atoms))
        variances.append(dyn.atoms.calc.results["energy_var"]/len(dyn.atoms))
        xtb_energies.append(xtb_energy/len(dyn.atoms))
        true_errors.append(np.var([dyn.atoms.calc.results["energy"],xtb_energy])/len(dyn.atoms))

        # plot the true error
        ax[0].plot(np.array(time_fs), np.array(true_errors), color="black")
        ax[0].set_ylabel(r'$\Delta$ E (eV$^2$/atom)')
        ax[0].legend(['Error w.r.t. xTB'], loc='upper left')

        # plot committee variance
        ax[1].plot(np.array(time_fs), np.array(variances), color="y")
        ax[1].set_ylabel(r'committee variance')
        ax[1].legend(['Estimated Error (committee variances)'], loc='upper left')

        # plot the temperature of the system as subplots
        ax[2].plot(np.array(time_fs), temperature, color="r", label='Temperature')
        ax[2].set_ylabel("T (K)")

        ax[3].plot(np.array(time_fs), energies_1, color="g")
        ax[3].plot(np.array(time_fs), energies_2, color="y")
        ax[3].plot(np.array(time_fs), energies_3, color="olive")
        ax[3].plot(np.array(time_fs), xtb_energies, color="black")
        ax[3].set_ylabel("E (eV/atom)")
        ax[3].set_xlabel('Time (fs)')
        ax[3].legend(['E mace1', 'E mace2', 'E mace3', 'E xtb'], loc='upper left')

        display.clear_output(wait=True)
        display.display(fig)
        time.sleep(0.01)

dyn.attach(write_frame, interval=10)
dyn.run(500)
print("MD finished!")

NOTE: if you get the error `xtb could not evalute the config` that means the dynamics went so crazy and gave such strange configurations, that xtb refused to run! thats to be expected if the model is really bad. Copy some of the code above to have a look at the trajectory if you'd like to see this.

Closely observe the dynamics. Notice how good the committee error is as a proxy for the true error. In this case the true is cheap to compute, but in most practical applications it won't be. Therefore, we will need to rely on the committee error to identy configurations that should be added back to the training set. This is called active learning:

![alt text](https://github.com/imagdau/Tutorials/blob/main/figures/active_learning.png?raw=1)

### Active learning in practice

The way to use active learning to improve the model is as follows:
1. run dynmics, track the uncertainty.
2. if the uncertainty breaches some predetermined value, stop the simulation and peform the ground truth calculation.
3. add the new config to the dataset, and retrain
4. repeat steps 1-3 until the uncertainty never crosses the threshold

This can be done without human supervision - you can write a program which loops this process.




**Task for you**
<div class="alert alert-block alert-info">


Write an active learing loop to gradually grow the dataset and produce a good model, without ever running xTB dynamics.
</div>

## 3 Foundational Models

### 3.1 Molecular Dynamics with MACE-MP-0

Foundation models changed everything. MACE-MP-0 is a model trained on >1 million DFT calcuations, and can run dynamics for the whole periodic table.

Mace provides a simple interface to load a foundational model, which we can use mow. Check the [documentation](https://mace-docs.readthedocs.io/en/latest/guide/foundation_models.html) for more details.

In [ ]:
from mace.calculators import mace_mp

macemp = mace_mp(model="medium-0b3",device=device)
init_conf = ea.sel_by_info_val(read(DATA_DIR / 'mace_data/solvent_molecs.xyz',':'), 'Nmols', 1)[0].copy()
simpleMD(init_conf, temp=1200, calc=macemp, fname=DATA_DIR / 'mace_moldyn/mace03_md.xyz', s=10, T=2000)

The dynamics looks good. It is stable with an out-of-the-box model. Let's inspect the trajectory:

In [ ]:
from weas_widget import WeasWidget

traj = read(DATA_DIR / 'mace_moldyn/mace03_md.xyz', ':')
w = WeasWidget()
w.from_ase(traj)
w.avr.model_style = 1
w

### 3.2 Compare to xTB

Let's compute the radial distribution functions of the trajectory and compare them to xTB. Remember MACE-MP was trained on PBE level of theory so we don't necessarily expect them to match:

In [ ]:
from matplotlib import pyplot as plt
from aseMolec import anaAtoms as aa

tag = 'CC_intra' #choose one of 'HH_intra', 'HC_intra', 'HO_intra', 'CC_intra', 'CO_intra', 'OO_intra'

for f in ['xtb_md', 'mace03_md']: # xtb_md was generated in the last workshop
    traj = read(DATA_DIR / 'mace_moldyn' / (f + '.xyz'), '50:') #ignore first 50 frames
    for at in traj:
        at.pbc = True #create a fake box for rdf compatibility
        at.cell = [100,100,100]
    rdf = aa.compute_rdfs_traj_avg(traj, rmax=3, nbins=200) #aseMolec provides functionality to compute RDFs
    plt.plot(rdf[1], rdf[0][tag], '.-', label=f, alpha=0.7, linewidth=3)

plt.legend();
plt.yticks([]);
plt.xlabel(r'R ($\rm \AA$)');
plt.ylabel('RDF '+tag);

Notice there's a substantial shift in the C-C RDF peak between xTB and MACE-MP-0. This is likely due to the different level of reference theory. Can we fix by fine tuning MACE-MP-0?

**Task for you**
<div class="alert alert-block alert-info">

Recompute the xTB energy on the MACE-MP-0 trajectory. How does it compare?
</div>

Depending on your application, the PBE reference used in MACE-MP-0 may not be appropriate, however it is already better than xTB. xTB was chosen here to minimize computational cost, so we can check the model's true error very easily.

In practice, you might want to run dynamics of this small molecule with a highly accurate electronic structure theory method. In that case, you would want to finetune MACE-MP-0 onto a small amount of very expensive data.

### 3.3 Fine tune MACE-MP to xTB

As described in the previous section, MACE-MP-0 offers qualitatively good performance across a wide range of chemistries and materials at the PBE+U level of theory.
However, for specific applications, it may be beneficial to fine-tune the model to improve its accuracy.
The fine-tuning process involves training a pre-trained model on a new dataset, called the fine-tuning dataset, which contains a limited amount of data that are only relevant to the specific application.

There exists three ways of finetuning a MACE model:
1. ***standard or naive*** approach which just restarts training using the parameters of the pretrained model
2. ***multi-head*** approach which ensures that the model does not forget too much about its pretraining.
3. ***frozen fine-tuning***, which restarts model training, but only updates a subset of the available weights and parameters during the fine-tuning. This also avoids forgetting and has the additional benefit that training is very fast as fewer parameters need updating.

We will start with the ***standard*** aproach below by setting `multiheads_finetuning: False` in the config. You will try for yourself to train another model with the ***multi-head*** approach and compare performances.

In [ ]:
%%writefile config/config-07.yml

model: "MACE"
stress_weight: 0.0
forces_weight: 10.0
energy_weight: 1.0
multiheads_finetuning: False
name: "finetuned_standard_MACE"
model_dir: "MACE_models"
log_dir: "MACE_models"
checkpoints_dir: "MACE_models"
results_dir: "MACE_models"
pt_train_file: mp
energy_key: "energy_xtb"
forces_key: "forces_xtb"
batch_size: 10
max_num_epochs: 100
stage_two: False
seed: 345
heads:
   xtb:


In [ ]:
#we now need to add the files to the config on which the 'xtb' head will be finetuned.

train_file = DATA_DIR / 'mace_data/solvent_xtb_train_20.xyz'
valid_file = DATA_DIR / 'mace_data/solvent_xtb_valid_20.xyz'
test_file = DATA_DIR / 'mace_data/solvent_xtb_test.xyz'
#unfortunately, the 8 spaces here matter as the yaml file has nested inputs for the 'xtb' head.
train = f'        train_file: {train_file}'
test =  f'        test_file: {test_file}'
valid = f'        valid_file: {valid_file}'

%store train >>"config/config-07.yml"
%store test >>"config/config-07.yml"
%store valid >>"config/config-07.yml"

dev = f'device: {device}'
%store dev >>"config/config-07.yml"

foundation_model_path = Path.home() / '.cache/mace/macemp0b3mediummodel' #path to the foundation model
found = f'foundation_model: {foundation_model_path}'
%store found >>"config/config-07.yml"


In [ ]:
train_mace("config/config-07.yml")

In [ ]:
from IPython.display import Image, display
display(Image("MACE_models/finetuned_standard_MACE_run-345_train_xtb_stage_one.png"))

Compare the final accuracy to the models trained from scratch. You should see that the errors are much better when doing finetuning.

In [ ]:
from IPython import display

mace_calc = MACECalculator(model_paths=['MACE_models/finetuned_standard_MACE_compiled.model'], device=device, dtype="float32")

init_conf = ea.sel_by_info_val(read(DATA_DIR / 'mace_data/solvent_molecs.xyz',':'), 'Nmols', 1)[0].copy()
simpleMD(init_conf, temp=1200, calc=mace_calc, fname=DATA_DIR / 'mace_moldyn/mace_finetuned_standard_md.xyz', s=10, T=2000)

In [ ]:
from matplotlib import pyplot as plt
from aseMolec import anaAtoms as aa

tag = 'CC_intra' # 'OO_intra' #choose one of 'HH_intra', 'HC_intra', 'HO_intra', 'CC_intra', 'CO_intra', 'OO_intra'

for f in ['xtb_md', 'mace_finetuned_standard_md']:
    traj = read(DATA_DIR / 'mace_moldyn' / (f + '.xyz'), '50:') #ignore first 50 frames
    for at in traj:
        at.pbc = True #create a fake box for rdf compatibility
        at.cell = [100,100,100]
    rdf = aa.compute_rdfs_traj_avg(traj, rmax=3, nbins=200)
    plt.plot(rdf[1], rdf[0][tag], '.-', label=f, alpha=0.7, linewidth=3)

plt.legend();
plt.yticks([]);
plt.xlabel(r'R ($\rm \AA$)');
plt.ylabel('RDF '+tag);

In [ ]:
from weas_widget import WeasWidget

traj = read(DATA_DIR / 'mace_moldyn' / 'mace_finetuned_standard_md.xyz', ':')
w = WeasWidget()
w.from_ase(traj)
w.avr.model_style = 1
w

What are the results - does it work?

**Tasks for you**
<div class="alert alert-block alert-info">

- Also compare the RDFS for the other element pairs
- Compare to the xTB energy. Does the fine-tuned model perform better than the model trained from scratch?
- Retry with the multi-head fine-tuning strategy, how does that perform?
- Note that the fine-tuned model has different hyperparameters inherited from the foundational model, is this a fair comparison? Train a model from scracth with the same parameters. How does it perform compared to the foundation model?
</div>

In [ ]:
%%writefile config/config-08.yml

model: "MACE"
stress_weight: 0.0
forces_weight: 10.0
energy_weight: 1.0
multiheads_finetuning: True
skip_evaluate_heads: False
name: "finetuned_multihead_MACE"
model_dir: "MACE_models"
log_dir: "MACE_models"
checkpoints_dir: "MACE_models"
results_dir: "MACE_models"
pt_train_file: mp

energy_key: "energy_xtb"
forces_key: "forces_xtb"
batch_size: 10
lr: 0.0001
scaling: rms_forces_scaling
force_mh_ft_lr: True
ema_decay: 0.99999
max_num_epochs: 50
num_samples_pt: 200
swa: False
seed: 345
heads:
   xtb:

In [ ]:
#we now need to add the files to the config on which the 'xtb' head will be finetuned.

train_file = DATA_DIR / 'mace_data/solvent_xtb_train_20.xyz'
valid_file = DATA_DIR / 'mace_data/solvent_xtb_valid_20.xyz'
test_file = DATA_DIR / 'mace_data/solvent_xtb_test.xyz'
#unfortunately, the 8 spaces here matter as the yaml file has nested inputs for the 'xtb' head.
train = f'        train_file: {train_file}'
test =  f'        test_file: {test_file}'
valid = f'        valid_file: {valid_file}'

%store train >>"config/config-08.yml"
%store test >>"config/config-08.yml"
%store valid >>"config/config-08.yml"

dev = f'device: {device}'
%store dev >>"config/config-08.yml"

foundation_model_path = Path.home() / '.cache/mace/macemp0b3mediummodel' #path to the foundation model
found = f'foundation_model: {foundation_model_path}'
%store found >>"config/config-08.yml"

In [ ]:
train_mace("config/config-08.yml")